In [1]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import json
import numpy as np
import os
import glob

from pathlib import Path, PosixPath
from typing import Dict, List, Tuple

In [2]:
def get_image_in_folder(path_to_image: str, suffix: List[str] = [".png", ".jpeg", ".jpg", ".tif", ".tiff"]) -> List[PosixPath]:
    """
    Search for image paths and image names.

    param:
        directory: str
            Directory for image search.

    returns:
        image_paths: list[PosixPath]
            List of image paths.
    """
    return [file_name for file_name in Path(path_to_image).rglob("*")
                   if file_name.suffix in suffix and ".ipynb_checkpoints" not in file_name.as_posix()]


def get_json_coordinate(path_json: str, preffix_image: str) -> Dict[str, List[str]]:
    """
   Get image coordinates

    Args:
        image_path: path to images
        camera_type: type of camera, "bottom" or "top"

    Returns:
        points: points for mapping
    """

    image_characteristic = {
        "image_main": [],
        "image_second": [],
        "points_main": [],
        "points_second": []
    }
    
    with open(path_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for idx in range(len(data)):
        points_main = []
        points_second = []
        
        variable_main = data[idx]["file1_path"].split('/')
        variable_second = data[idx]["file2_path"].split('/')
        
        image_characteristic["image_main"].append(f"{preffix_image}/{variable_main[-3]}/{variable_main[-2]}/{variable_main[-1]}")
        image_characteristic["image_second"].append(f"{preffix_image}/{variable_second[-3]}/{variable_second[-2]}/{variable_second[-1]}")
        
        for p in zip(data[idx]["image1_coordinates"], data[idx]["image2_coordinates"]):
            points_main.append([p[0]["x"], p[0]["y"]])
            points_second.append([p[1]["x"], p[1]["y"]])
            
        image_characteristic["points_main"].append(np.array(points_main))
        image_characteristic["points_second"].append(np.array(points_second))

    return image_characteristic

In [3]:
from solution.train import HomographyTrain

In [4]:
homogr = HomographyTrain('bottom')

In [5]:
train_json = get_image_in_folder("train", suffix = [".json"])
val_json = get_image_in_folder("val", suffix = [".json"])
homogr.train(train_json, val_json)


Обучение для камеры: bottom
Тренировочных точек: 3369
Валидационных точек: 970

Производится расчёт матрицы гомографии (улучшенный метод)...
  RANSAC (порог=3.0): inlier_ratio = 99.85%
  RANSAC (порог=5.0): inlier_ratio = 100.00%
  LMEDS: inlier_ratio = 99.79%
  RHO (порог=3.0): inlier_ratio = 100.00%
Выбран метод: RANSAC (порог=5.0) (inlier_ratio=100.00%)
Матрица гомографии найдена: [[-2.58872475e-01 -2.12812811e-01  2.54811967e+03]
 [ 6.17375460e-01  1.47086845e+00 -3.62869554e+02]
 [-1.66355693e-04  1.77841515e-03  1.00000000e+00]]

==================== Расчёт метрик для Train сплита ====================
Среднее евклидовое расстояние (MED): 196.43 px
Медианное евклидовое расстояние: 157.51 px
Стандартное отклонение: 156.66 px
95-й процентиль: 494.68 px
Максимальная ошибка: 1385.02 px

==================== Расчёт метрик для Val сплита ====================
Среднее евклидовое расстояние (MED): 193.91 px
Медианное евклидовое расстояние: 153.25 px
Стандартное отклонение: 154.33 px
95-й 

In [6]:
homogr_top = HomographyTrain('top')
train_json = get_image_in_folder("train", suffix = [".json"])
val_json = get_image_in_folder("val", suffix = [".json"])
homogr_top.train(train_json, val_json)


Обучение для камеры: top
Тренировочных точек: 3915
Валидационных точек: 1034

Производится расчёт матрицы гомографии (улучшенный метод)...
  RANSAC (порог=3.0): inlier_ratio = 99.36%
  RANSAC (порог=5.0): inlier_ratio = 100.00%
  LMEDS: inlier_ratio = 96.63%
  RHO (порог=3.0): inlier_ratio = 99.57%
Выбран метод: RANSAC (порог=5.0) (inlier_ratio=100.00%)
Матрица гомографии найдена: [[ 1.42598796e+00  1.09900325e+01 -2.09353967e+03]
 [-1.32953259e+00  2.47011032e+00  3.88662236e+03]
 [ 7.59326122e-04  3.23596737e-03  1.00000000e+00]]

==================== Расчёт метрик для Train сплита ====================
Среднее евклидовое расстояние (MED): 238.52 px
Медианное евклидовое расстояние: 175.84 px
Стандартное отклонение: 223.93 px
95-й процентиль: 598.35 px
Максимальная ошибка: 2405.23 px

==================== Расчёт метрик для Val сплита ====================
Среднее евклидовое расстояние (MED): 205.08 px
Медианное евклидовое расстояние: 154.51 px
Стандартное отклонение: 154.09 px
95-й про

In [7]:
from solution.predict import HomographyMapper

In [8]:
all_image_characteristic = {}

for jsn in train_json:
    if 'bottom' in jsn.as_posix():
        image_characteristic = get_json_coordinate(jsn, "train")
        all_image_characteristic[jsn] = image_characteristic

In [9]:
hmapper = HomographyMapper('solution/Homography.json')
hmapper.predict(all_image_characteristic[train_json[1]]['points_second'][0][0][0],
        all_image_characteristic[train_json[1]]['points_second'][0][0][1],
       "bottom")

Loaded homography for bottom camera
Loaded homography for top camera


(687.1155395507812, 470.17242431640625)

In [11]:
from solution.train import HomographyTrain

In [10]:
homogr = HomographyTrain('bottom', True)

In [11]:
train_json = get_image_in_folder("train", suffix = [".json"])
val_json = get_image_in_folder("val", suffix = [".json"])
homogr.train(train_json, val_json)


Обучение для камеры: bottom
Тренировочных точек: 3369
Валидационных точек: 970

Производится обучение MLP регрессора для нелинейного маппинга...
MLP модель обучена. Итераций: 69, потери: 0.062619

==================== Расчёт метрик для Train сплита ====================
Среднее евклидовое расстояние (MED): 145.07 px
Медианное евклидовое расстояние: 104.25 px
Стандартное отклонение: 136.41 px
95-й процентиль: 416.85 px
Максимальная ошибка: 1473.45 px

==================== Расчёт метрик для Val сплита ====================
Среднее евклидовое расстояние (MED): 143.78 px
Медианное евклидовое расстояние: 106.18 px
Стандартное отклонение: 132.47 px
95-й процентиль: 378.74 px
Максимальная ошибка: 1375.82 px

==================== Визуализация результата для Train ====================
Результат сохранён по пути: result/bottom_train_merged.jpg

==================== Визуализация результата для Val ====================
Результат сохранён по пути: result/bottom_val_merged.jpg
ML модель сохранена в s